# Выполнение ЛР №6: "Поиск ассоциативных правил" 

## Подключение библиотек

In [ ]:
import os
import pandas as pd
import numpy as np
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules

## Настройка библиотек

In [2]:
pd.set_option('display.max_rows', None)

## Задание

Реализовать  на  любом языке программирования алгоритм поиска ассоциативных правил: Apriori

Для  проверки  корректности 
алгоритма на любом предлагаемом датасете предлагается 
применить библиотеку mlxtend  языка Python.

## Изучение датасета

In [ ]:
# Загрузка датасета Anketa1

path_anketa = os.path.join(os.getcwd(), 'Задание', 'dataset', 'Anketa1.txt')
df_raw = pd.read_csv(path_anketa, sep='\t', decimal=',', encoding='cp1251', encoding_errors='replace')
df_raw.head(10)

In [ ]:
# Разведка данных
print('Размер:', df_raw.shape)
print('\nТипы и пропуски:')
print(df_raw.dtypes)

In [ ]:
# Уникальные значения категориальных признаков (для бинаризации)
cat_cols = [
    'Социальный статус',
    'Социальный статус супруга(и)',
    'Образование',
    'Наличие личного автомобиля',
    'Жилая недвижимость в собственности',
    'Наличие кредитов'
]
cat_uniques = {col: df_raw[col].astype(str).unique() for col in cat_cols}
df_cat_uniques = pd.DataFrame(dict([(col, pd.Series(vals)) for col, vals in cat_uniques.items()]))
display(df_cat_uniques)

## Подготовка DataFrame для ассоциативных правил

In [ ]:
# Удаляем идентификаторы
cols_drop = ['КодАнкеты', 'Фамилия', 'Имя', 'Отчество']
df = df_raw.drop(columns=[c for c in cols_drop if c in df_raw.columns], errors='ignore')

# Унификация да/нет (приведём к одному регистру)
binary_cols = ['Наличие личного автомобиля', 'Наличие кредитов']
for col in binary_cols:
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip().str.lower()
        df.loc[df[col].str.contains('да', na=False), col] = 'да'
        df.loc[df[col].str.contains('нет', na=False), col] = 'нет'

# Недвижимость: много уникальных значений — оставляем как есть или группируем
if 'Жилая недвижимость в собственности' in df.columns:
    df['Жилая недвижимость в собственности'] = df['Жилая недвижимость в собственности'].astype(str).str.strip()
df.head()

In [ ]:
# Дискретизация числовых признаков (квартили или фиксированные границы)
def discretize(s, n_bins=3, labels=None):
    if labels is None:
        labels = [f'Q{i+1}' for i in range(n_bins)]
    try:
        # без labels — узнаём реальное число интервалов
        binned = pd.qcut(s, q=n_bins, duplicates='drop')
        n_actual = binned.cat.categories.size
        use_labels = labels[:n_actual] if len(labels) >= n_actual else [f'Q{i+1}' for i in range(n_actual)]
        return pd.qcut(s, q=n_bins, labels=use_labels, duplicates='drop')
    except Exception:
        # если не получилось — возвращаем без подписей (категории 0,1,...)
        return pd.qcut(s, q=n_bins, duplicates='drop')

num_cols = {
    'Сумма кредита, руб#': ['Кредит_низ', 'Кредит_сред', 'Кредит_выс'],
    'Количество лет проживания в регионе': ['Регион_мало', 'Регион_сред', 'Регион_много'],
    'Стаж работы, лет': ['Стаж_мало', 'Стаж_сред', 'Стаж_много'],
    'Личный доход в месяц после налогооблажения': ['Доход_низ', 'Доход_сред', 'Доход_выс'],
    'Рыночная стоимость автомобиля, руб#': ['Авто_нет_низ', 'Авто_сред', 'Авто_выс'],  # 0 попадает в низ
    'Рыночная стоимость недвижимости, руб#': ['Недвиж_нет_низ', 'Недвиж_сред', 'Недвиж_выс'],
}

for col, lab in num_cols.items():
    if col in df.columns:
        try:
            df[col + '_кат'] = discretize(df[col].astype(float), n_bins=len(lab), labels=lab)
        except Exception as e:
            print(col, e)
            
df.head(10)

In [ ]:
# Собираем все категориальные колонки для транзакций
cat_cols = [
    'Социальный статус', 'Социальный статус супруга(и)', 'Образование',
    'Наличие личного автомобиля', 'Жилая недвижимость в собственности', 'Наличие кредитов',
    'Возврат кредита'
]
# Добавляем дискретизированные
cat_cols += [c for c in df.columns if c.endswith('_кат')]

# Удаляем дубликаты и несуществующие
cat_cols = [c for c in cat_cols if c in df.columns]

df_cat = df[cat_cols].astype(str)
df_cat.head()

In [ ]:
# Преобразование в формат "транзакций" для mlxtend: одна колонка = один предмет (признак=значение)
# Каждая строка — одна анкета (транзакция), колонки — бинарные признаки вида "Образование=высшее"
transactions = []
for _, row in df_cat.iterrows():
    trans = [f"{col}={val}" for col, val in row.items() if pd.notna(val) and str(val).strip() and str(val) != 'nan']
    transactions.append(trans)

te = TransactionEncoder()
te_ary = te.fit(transactions).transform(transactions)
df_transactions = pd.DataFrame(te_ary, columns=te.columns_).astype(int) 
print('Размер матрицы транзакций:', df_transactions.shape)
df_transactions.head(10)